# Google Colab Bluebird Generation

Use this notebook on a Colab GPU runtime, ideally a T4. It reuses the existing local generation script, but with shorter token counts so you are not waiting on a huge autoregressive run.

Speed knobs to try: lower `IBP_MAX_NEW_TOKENS` first, then raise `IBP_NUM_SAMPLES` if you want to batch multiple candidates on the GPU.

In [ ]:
from pathlib import Path
import sys

IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

def _is_repo_root(path: Path) -> bool:
    return (
        (path / 'local_generate_once.py').exists()
        and (path / 'app' / 'models').exists()
        and (path / 'app' / 'tokenizer').exists()
    )

PROJECT_ROOT_CANDIDATES = [
    Path('/content/piano_midi_model'),
    Path('/content/drive/MyDrive/piano_midi_model'),
    Path('/content/drive/MyDrive/piano_model/piano_midi_model'),
    Path('/content/drive/MyDrive/midi_AI/piano_midi_model'),
    Path.cwd().resolve(),
]

PROJECT_ROOT = next((candidate for candidate in PROJECT_ROOT_CANDIDATES if _is_repo_root(candidate)), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find the repository root. Put the repo in /content/piano_midi_model or mount it under Google Drive.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'IN_COLAB={IN_COLAB}')
print(f'PROJECT_ROOT={PROJECT_ROOT}')

In [ ]:
import subprocess
import sys

req_candidates = [
    PROJECT_ROOT / ('requirements_colab.txt' if IN_COLAB else 'requirements.txt'),
    PROJECT_ROOT / 'app' / 'requirements.txt',
]
req_file = next((path for path in req_candidates if path.exists()), None)
if req_file is None:
    raise FileNotFoundError('Could not find requirements file in the repository.')

print(f'Installing dependencies from {req_file}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], check=True)

In [ ]:
import os
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

seed_candidates = [
    PROJECT_ROOT / 'bluebird.mid',
    PROJECT_ROOT / 'app' / 'runtime' / 'bluebird.mid',
    Path('/content/bluebird.mid'),
]
SEED_PATH = next((path for path in seed_candidates if path.exists()), None)
if SEED_PATH is None:
    raise FileNotFoundError('Could not find bluebird.mid in the repo or notebook working directory.')

# Keep the sequence shorter first. Increase this if you want a longer continuation.
os.environ['IBP_SEED_PATH'] = str(SEED_PATH)
os.environ['IBP_MAX_NEW_TOKENS'] = '384'
os.environ['IBP_NUM_SAMPLES'] = '1'
os.environ['IBP_TEMPERATURE'] = '0.90'
os.environ['IBP_TOP_P'] = '0.95'
os.environ['IBP_TOP_K'] = '50'

print(f'SEED_PATH={SEED_PATH}')
print('Tip: set IBP_NUM_SAMPLES=2 or 4 if you want batched GPU samples.')

In [ ]:
import subprocess
import sys

script_path = PROJECT_ROOT / 'local_generate_once.py'
print(f'Running {script_path}')
subprocess.run([sys.executable, str(script_path)], cwd=str(PROJECT_ROOT), check=True)

In [ ]:
from IPython.display import Audio, Image, display

OUTPUT_DIR = PROJECT_ROOT / 'app' / 'runtime' / 'outputs'
stem = Path(os.environ['IBP_SEED_PATH']).stem + '_continuation'
midi_path = OUTPUT_DIR / f'{stem}.mid'
seed_audio_path = OUTPUT_DIR / f'{stem}_seed_preview.wav'
output_audio_path = OUTPUT_DIR / f'{stem}_output_preview.wav'
comparison_path = OUTPUT_DIR / f'{stem}_comparison.png'

print(f'MIDI: {midi_path}')
print(f'Seed audio: {seed_audio_path}')
print(f'Output audio: {output_audio_path}')
print(f'Comparison: {comparison_path}')

if seed_audio_path.exists():
    display(Audio(filename=str(seed_audio_path)))
if output_audio_path.exists():
    display(Audio(filename=str(output_audio_path)))
if comparison_path.exists():
    display(Image(filename=str(comparison_path)))